# Digital Cash Management Forecast: lab (data scientist)

Explore the data and try models here without touching production: nothing in this notebook changes the flow, the
saved models or the production model.

1. **EDA**: the daily flows per category, totals over time, seasonality, and one client day by day.
2. **Models to try**: several forecasters trained and tested side by side on the same test as champion / challenger
   (trained without the last 6 periods of 30 days before the cut-off, tested on them), next to the production model
   and the simple guess (the same as the 30 days before), with the decision champion / challenger would take for each.
   Each try is recorded in Experiment Tracking, experiment `cashflow_lab` (same folder as the production models,
   `experiments`, but its own experiment); lab tries are never deployed.
3. **Put one in production.**

**Kernel:** the code env `cashflow_forecast`. If it is not in the kernel list, ask whoever manages it to enable its
Jupyter support. Meanwhile a forecaster can be tried safely through **Retrain, test and promote**: it goes live
only if it beats the production model.

In the sandbox the data is simulated: there, the scores show that the code works, not which model is better.

In [ ]:
import calendar
import importlib
import logging
import warnings

warnings.filterwarnings("ignore", category=FutureWarning, module="google")      # third-party notices on import
logging.getLogger("matplotlib").setLevel(logging.ERROR)
try:
    import mlflow
    import sklearn
except ImportError:
    raise RuntimeError("Use the kernel of the code env cashflow_forecast (Kernel > Change kernel). If it is not in the "
                       "list, ask whoever manages that code env to enable its Jupyter support. Meanwhile a forecaster "
                       "can be tried safely through Retrain: it goes live only if it beats the production model.")
import dataiku

warnings.filterwarnings("ignore", category=DeprecationWarning)    # third-party notices; after dataiku, which shows them
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import cashflow
import forecasters
import models
import tracking

for module in (cashflow, models, tracking, forecasters):     # pick up changes to the project library without a restart
    importlib.reload(module)
from cashflow import (FORECAST_COLUMNS, HORIZONS, PERIOD_DAYS, TOTAL, cutoff_date, monthly_series, period_matrix,
                      read_daily, read_dataset)
from forecasters import RecentAverage6
from models import QuantileBoosting, RecentAverage, lab_compare

pd.set_option("display.max_columns", 30)
cutoff = cutoff_date()
daily = read_daily()                         # interface 1: one row per client, day and category
clients = read_dataset("clients")            # one row per business client
monthly = monthly_series(daily, cutoff)      # the net cash flow per client, category (and TOTAL) and month
print(f"daily_flows: {len(daily):,} rows, {daily['client_no'].nunique()} clients, {daily['category'].nunique()} "
      f"categories, {pd.to_datetime(daily['day']).min():%Y-%m-%d} to {pd.to_datetime(daily['day']).max():%Y-%m-%d}; "
      f"cut-off day {cutoff:%Y-%m-%d}")

## 1. EDA

The data every model gets (interface 1): `daily_flows`, one row per client, day and category with money in and
money out, and `clients`. The models predict, per client, per category and in TOTAL (all categories together), the
net cash flow (money in minus money out) of each of the next 30, 60 and 90 days. First, per category and per segment
(**negative_months**: the share of months where more money went out than came in).

In [ ]:
# Per category: how many clients use it, money in and out per month, and how often a client's month is negative
d = daily.assign(month=cashflow.month_start(daily["day"]).values)
per_month = d.groupby(["category", "client_no", "month"])[["money_in", "money_out"]].sum().reset_index()
overview = per_month.groupby("category").agg(clients=("client_no", "nunique"), money_in=("money_in", "mean"),
                                             money_out=("money_out", "mean"))
overview["negative_months"] = (monthly[monthly["category"] != TOTAL].assign(neg=lambda m: m["net"] < 0)
                               .groupby("category")["neg"].mean())
display(overview.round({"money_in": 0, "money_out": 0, "negative_months": 3}).sort_values("clients", ascending=False))

# Per segment, the client's whole net cash flow (TOTAL)
total = monthly[monthly["category"] == TOTAL].merge(clients[["client_no", "segment"]], on="client_no")
display(total.groupby("segment").agg(clients=("client_no", "nunique"), months=("month", "size"),
                                     net=("net", "mean"), negative_months=("net", lambda s: (s < 0).mean()))
        .round({"net": 0, "negative_months": 3}))

Net cash flow per category over time, and seasonality. Ramadan and Eid move about 11 days earlier each year, so a
calendar-month profile blurs them.

In [ ]:
# All clients per month: the net cash flow of each category (million SAR)
by_category = (monthly[monthly["category"] != TOTAL].pivot_table(index="month", columns="category", values="net",
                                                                  aggfunc="sum") / 1e6)
ax = by_category.plot(figsize=(12, 4.5), title="Net cash flow per category, all clients (million SAR)")
ax.axhline(0, color="grey", linewidth=0.5)
ax.set_xlabel("")
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))
plt.tight_layout()
plt.show()

# Seasonality without the trend: money in and out each month / the average of the 12 months before (1 = usual)
d = daily.assign(month=cashflow.month_start(daily["day"]).values)
totals = d.groupby("month")[["money_in", "money_out"]].sum()
ratio = totals / totals.rolling(12).mean().shift(1)
profile = ratio.groupby(ratio.index.month).mean()
profile.index = [calendar.month_abbr[m] for m in profile.index]
ax = profile.plot.bar(figsize=(10, 3.5), title="Seasonality: month / the 12 months before, all clients (1 = usual)")
ax.axhline(1, color="grey", linewidth=0.5)
plt.show()

One client day by day (what the models see) and its monthly totals.

In [ ]:
# One client, day by day: what the models see (daily net, all categories), and the monthly totals they predict
client_no = monthly[monthly["category"] == TOTAL].groupby("client_no")["net"].size().idxmax()   # longest history
one = daily[daily["client_no"] == client_no].assign(net=lambda d: d["money_in"] - d["money_out"])
one = one.groupby(pd.to_datetime(one["day"]))["net"].sum().loc[lambda s: s.index > cutoff - pd.DateOffset(months=6)]
fig, (a, b) = plt.subplots(1, 2, figsize=(13, 3.5))
(one / 1e3).plot(ax=a, style=".", title=f"Client {client_no}: daily net, last 6 months (thousand SAR)")
months = monthly[(monthly["client_no"] == client_no) & (monthly["category"] == TOTAL)].set_index("month")["net"]
(months.tail(24) / 1e3).plot.bar(ax=b, title="Monthly net, last 24 months (thousand SAR)")
b.set_xticklabels([f"{m:%y-%m}" for m in months.tail(24).index], rotation=90)
plt.tight_layout()
plt.show()

## 2. Models to try

A forecaster is a class with `name`, `params` (its settings), `fit(daily, clients, cutoff)` and
`predict(daily, clients, cutoff)`, which returns one row per client, category (and TOTAL) and window (1, 2, 3: days
1-30, 31-60, 61-90 after the cut-off day) with `p10`, `ev` and `p90` in SAR (details in `models.py`, project library).
`cashflow.period_matrix(daily, cutoff)` turns the days into 30-day periods ending at the cut-off day. Compared here:

- `quantile_boosting`: the placeholder in production. 3 small models (scikit-learn gradient boosting on the last 12
  periods of 30 days): one for the EV, one for P10, one for P90. Each is shared by every category and TOTAL, with the category as
  an input.
- `quantile_boosting_by_category`: the other way to split it, defined below: a separate set of 3 models for each
  category and TOTAL, each trained only on its own category.
- `recent_average`: no machine learning (the average of the last 3 periods of 30 days), and `recent_average_6m` (over
  6 periods, an example in `forecasters.py`)

Change them or add your own, then run the comparison. scikit-learn, pandas and matplotlib are in the code env; other
packages need the admin.

In [ ]:
class QuantileBoostingByCategory:
    """The other way to split the work: a separate set of 3 models (EV, P10, P90) for each category and TOTAL, each
    trained only on its own category. The production placeholder (QuantileBoosting) has one set for all categories,
    with the category as an input. A category the training data did not have uses the shared set."""
    name = "quantile_boosting_by_category"
    params = dict(QuantileBoosting.params)

    def fit(self, daily, clients, cutoff):
        wide = period_matrix(daily, cutoff)
        categories = wide.index.get_level_values("category")
        self.shared = QuantileBoosting().fit_periods(wide)
        self.models = {c: QuantileBoosting().fit_periods(wide[categories == c]) for c in sorted(set(categories))}
        return self

    def predict(self, daily, clients, cutoff):
        wide = period_matrix(daily, cutoff)
        categories = wide.index.get_level_values("category")
        return pd.concat([self.models.get(c, self.shared).predict_periods(wide[categories == c])
                          for c in sorted(set(categories))], ignore_index=True)

In [ ]:
# Train each on the same months, test it on the last 6, next to the production model. quantile_boosting is the
# production recipe trained again: when the production model was trained on the same cut-off day, their scores match
# (a check that the test is the same).
RECORD = True       # record each try in Experiment Tracking (experiment cashflow_lab)
result = lab_compare([QuantileBoosting(), QuantileBoostingByCategory(), RecentAverage(), RecentAverage6()],
                     record=RECORD)
print("Net cash flow per client, category and TOTAL. EV error: lower is better. P10-P90 coverage: aim 80%.")
display(result["board"])
print("EV error per window (ALL: together):")
display(result["errors"])
print("P10-P90 coverage per window:")
display(result["coverage"])

In [ ]:
errors = result["errors"].drop(index="ALL")
ax = errors.plot.bar(figsize=(10, 4), title="EV error per window (lower is better)")
ax.set_xlabel("window (1 = days 1-30, 2 = 31-60, 3 = 61-90)")
plt.tight_layout()
plt.show()

## 3. Put one in production

1. Copy its class into `forecasters.py` (project library) with the imports it needs (for example
   `import pandas as pd`, `from cashflow import FORECAST_COLUMNS, HORIZONS, TOTAL, period_matrix`) and add it to
   `FORECASTERS`. The setup notebook never overwrites that file.
2. Set the project variable `forecaster` to its name.
3. Run the scenario **Retrain, test and promote**. Champion / challenger promotes it only if it passes the
   promotion rules (project variables `promotion_*`) against the production model on the same 6 periods; the next
   Nightly batch forecasts with it.
4. To stop training it, set `forecaster` back to `quantile_boosting`. To use an earlier model now, activate its
   version in the saved model `cash_forecast`.